# 02. Tool Executor — Pipeline & Sandbox

`arcrun.executor.execute_tool_call()` is the single shared pipeline every strategy
calls when a model emits a tool call. It is deliberately small (~85 lines) because
*it has one job*: take a tool call from the model, walk it through permission
checks, validation, execution, and audit emission, and hand back a result message
the strategy can append to the conversation.

This notebook covers the surface area around that pipeline:

- The **`ToolRegistry`** — defining tools, registering, listing, replacing, removing, dynamic mid-run updates
- **`Tool` and `ToolContext`** — what a tool *is* and what it sees at runtime
- **`SandboxConfig`** — the permission boundary, in two modes: allowlist and custom checker
- **`SandboxError`** taxonomy — every exception subtype, when each fires, with intentional raises you can read the trace of
- **Classification** — `classification` on `Tool` and the `ClassificationRegistry` Protocol (`read_only` vs `state_modifying`)
- **Errors, timeouts, retries** — what happens when a tool raises, how the LLM sees it, what the audit log keeps

Two adjacent topics get their own notebooks:

- **Parallel dispatch deep dive** → `arcrun/05-parallel-dispatch.ipynb`
- **Container execution (Docker isolation)** → `arcrun/03-codeexec.ipynb`

Everything here runs end-to-end without an API key. We use a tiny inline
`MockModel` instead of touching a real provider.

## 1. Setup

The standard arc-walkthrough boilerplate. It locates the repo root by walking
upward looking for `packages/` and `pyproject.toml`, then puts every package's
`src/` (and `tests/`, where present) on `sys.path`. After that, `from arcrun ...`
just works.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

### Quick smoke test of the imports

If this runs clean, the path wiring is correct.

In [2]:
import arcrun

print(f"arcrun {arcrun.__version__}")
print()
public = sorted(s for s in arcrun.__all__ if not s.startswith("_"))
print("Public symbols on the executor surface:")
for sym in (
    "ToolRegistry",
    "Tool",
    "ToolContext",
    "SandboxConfig",
    "SandboxError",
    "SandboxOOMError",
    "SandboxRuntimeError",
    "SandboxTimeoutError",
    "SandboxUnavailableError",
):
    print(f"  {sym:30s} present={sym in public}")

arcrun 0.9.0

Public symbols on the executor surface:
  ToolRegistry                   present=True
  Tool                           present=True
  ToolContext                    present=True
  SandboxConfig                  present=True
  SandboxError                   present=True
  SandboxOOMError                present=True
  SandboxRuntimeError            present=True
  SandboxTimeoutError            present=True
  SandboxUnavailableError        present=True


## 2. Tool execution pipeline overview

When a model emits `tool_calls=[ToolCall(name="read_file", arguments={...})]`,
the strategy (ReAct, CodeExec, etc.) does **not** execute it inline. It hands
each call to the shared executor. That executor is the *only* place in arcrun
that runs a tool — every strategy gets the same sandbox, schema, audit, and
timeout behavior for free.

```
   model.invoke()                        run loop / strategy
        │                                       │
        ▼                                       │
   [tool_calls]   ─────────────────────────────►│
                                                │
                                                ▼
                              ┌─────────────────────────────────┐
                              │  execute_tool_call(tc, state,   │
                              │                    sandbox)     │
                              │                                 │
                              │   1. emit tool.start            │
                              │   2. sandbox.check()            │  ──► tool.denied
                              │   3. registry.get(name)         │  ──► "tool not found"
                              │   4. jsonschema.validate(args)  │  ──► "invalid params"
                              │   5. build ToolContext          │
                              │   6. await tool.execute(...)    │  ──► tool.error (raise/timeout)
                              │   7. emit tool.end              │
                              │   8. state.tool_calls_made += 1 │
                              │   9. return (Message, True)     │
                              └─────────────────────────────────┘
                                                │
                                                ▼
                                  Message(role="tool", ...)
                                  appended back to conversation
```

The contract is small: **`(tool_call, RunState, Sandbox) -> (tool_result_message, success_bool)`**.
The strategy never sees jsonschema. It never sees sandbox config. It just calls
the executor and appends the message to `state.messages`.

## 3. The `Tool` dataclass and `ToolContext`

A `Tool` is five fields. The model sees `name`, `description`, and `input_schema`.
The executor calls `execute(params, ctx)`. `timeout_seconds` is optional; if unset,
the executor falls back to `state.tool_timeout`.

```python
@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict[str, Any]
    execute: Callable[[dict, ToolContext], Awaitable[str]]
    timeout_seconds: float | None = None
    classification: str = "state_modifying"  # "read_only" tools may run via asyncio.gather
    signals_completion: bool = False    # an invocation terminates the loop
```

Let's define a couple of small synthetic tools (no I/O — pure functions of their
arguments) and look at what `ToolContext` carries.

In [3]:
from arcrun.types import Tool


async def _search_execute(params, ctx):
    # ctx exposes run_id, tool_call_id, turn_number, the EventBus, and a
    # cancelled signal — but a tool doesn't have to use any of it.
    return f"Found 3 results for '{params['query']}'"


search_tool = Tool(
    name="search",
    description="Search the local index",
    input_schema={
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
    execute=_search_execute,
    classification="read_only",  # safe to run alongside itself
)

print(f"name              : {search_tool.name}")
print(f"description       : {search_tool.description}")
print(f"input_schema      : {search_tool.input_schema}")
print(f"timeout_seconds   : {search_tool.timeout_seconds}  (None = use state default)")
print(f"classification     : {search_tool.classification}")
print(f"signals_completion: {search_tool.signals_completion}")

name              : search
description       : Search the local index
input_schema      : {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
timeout_seconds   : None  (None = use state default)
classification     : read_only
signals_completion: False


### `ToolContext` — what the tool sees at runtime

The executor builds a fresh `ToolContext` for every tool call:

| Field | Purpose |
|---|---|
| `run_id` | The current run's identifier — useful for tagging downstream events |
| `tool_call_id` | The model-generated id for *this* call |
| `turn_number` | 1-indexed turn we're in |
| `event_bus` | The shared `EventBus` — tools can emit custom events for visibility |
| `cancelled` | An `asyncio.Event` — long-running tools should poll this and bail |
| `parent_state` | Live `RunState` — exposed so tools (e.g., `delegate`) can read depth/budget |

Most tools ignore most of it. Long-running ones should at minimum check `ctx.cancelled.is_set()` periodically.

In [4]:
import asyncio

from arcrun.types import ToolContext

ctx = ToolContext(
    run_id="demo-001",
    tool_call_id="tc-42",
    turn_number=3,
    event_bus=None,
    cancelled=asyncio.Event(),
    parent_state=None,
)
print(f"run_id        : {ctx.run_id}")
print(f"tool_call_id  : {ctx.tool_call_id}")
print(f"turn_number   : {ctx.turn_number}")
print(f"cancelled.set : {ctx.cancelled.is_set()}")

run_id        : demo-001
tool_call_id  : tc-42
turn_number   : 3
cancelled.set : False


### Inline mocks — why no `from conftest import ...`

`packages/arcrun/tests/` is on `sys.path`, which means a `from conftest import ...`
would shadow pytest's conftest discovery. Safer to define the few stubs we need
right here in the notebook. Real arcllm types still drive the executor (`Message`,
`ToolResultBlock`); we only stub the model-facing `LLMResponse` / `MockModel`.

In [5]:
from dataclasses import dataclass, field
from typing import Any

from arcllm.types import Message, ToolCall  # real Pydantic types


@dataclass
class Usage:
    input_tokens: int = 10
    output_tokens: int = 5
    total_tokens: int = 15


@dataclass
class LLMResponse:
    content: str | None = None
    tool_calls: list[ToolCall] = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: Usage = field(default_factory=Usage)
    cost_usd: float = 0.001


class MockModel:
    """Replays a scripted list of LLMResponse objects, one per invoke()."""

    def __init__(self, responses: list[LLMResponse]) -> None:
        self._responses = list(responses)
        self._idx = 0

    async def invoke(self, messages: list, tools: list | None = None) -> LLMResponse:
        if self._idx >= len(self._responses):
            return LLMResponse(content="(mock exhausted)", stop_reason="end_turn")
        resp = self._responses[self._idx]
        self._idx += 1
        return resp


print("Mocks ready.")

Mocks ready.


## 4. `ToolRegistry` — the mutable tool collection

`ToolRegistry` is what the executor reaches into when it needs to look up a tool
by name. It is **mutable on purpose** — strategies and extensions can add or
remove tools mid-run, and the model will see the new tool list on its next turn
(`registry.list_schemas()` is called every loop iteration).

The full public surface:

| Method | Purpose | Event emitted |
|---|---|---|
| `add(tool)` | Register or replace a tool by name | `tool.registered` (new) or `tool.replaced` (existing) |
| `remove(name)` | Drop a tool. No-op if absent | `tool.removed` (only if found) |
| `get(name) -> Tool \| None` | Lookup by name | — |
| `list_schemas() -> list[arcllm.Tool]` | Convert to model-facing schemas | — |
| `names() -> list[str]` | Sorted list of registered names | — |

In [6]:
from arcrun.events import EventBus
from arcrun.registry import ToolRegistry

bus = EventBus(run_id="reg-demo")
registry = ToolRegistry(tools=[search_tool], event_bus=bus)

print(f"names()           : {registry.names()}")
print(f"get('search')     : {registry.get('search').name}")
print(f"get('missing')    : {registry.get('missing')}")
print(f"len(list_schemas): {len(registry.list_schemas())}")

names()           : ['search']
get('search')     : search
get('missing')    : None
len(list_schemas): 1


### Adding a tool dynamically

Every mutation emits an event — that's how a UI ("agent just gained a `write_file`
tool!") or an audit log gets to know about it without polling.

In [7]:
async def _write_execute(params, ctx):
    return f"wrote to {params['path']}"


write_tool = Tool(
    name="write_file",
    description="Write bytes to a file",
    input_schema={
        "type": "object",
        "properties": {"path": {"type": "string"}},
        "required": ["path"],
    },
    execute=_write_execute,
    classification="state_modifying",  # writes mutate state — must be serial
)

registry.add(write_tool)
print(f"After add  : {registry.names()}")

# Replacing — emits tool.replaced, not tool.registered
write_v2 = Tool(
    name="write_file",
    description="Write bytes (v2 — atomic)",
    input_schema=write_tool.input_schema,
    execute=_write_execute,
)
registry.add(write_v2)
print(f"After replace: get('write_file').description = {registry.get('write_file').description!r}")

print()
print("Events emitted by the registry:")
for e in bus.events:
    print(f"  {e.type:18s} {dict(e.data)}")

After add  : ['search', 'write_file']
After replace: get('write_file').description = 'Write bytes (v2 — atomic)'

Events emitted by the registry:
  tool.registered    {'name': 'write_file'}
  tool.replaced      {'name': 'write_file'}


### Removing a tool

`remove()` is a no-op if the name isn't present — no exception. That's by design:
strategies and extensions can call it idempotently.

In [8]:
registry.remove("write_file")
registry.remove("ghost_that_was_never_registered")  # silent no-op
print(f"After removes: {registry.names()}")
print()
print("Events from this cell:")
for e in bus.events[-3:]:
    print(f"  {e.type:18s} {dict(e.data)}")

After removes: ['search']

Events from this cell:
  tool.registered    {'name': 'write_file'}
  tool.replaced      {'name': 'write_file'}
  tool.removed       {'name': 'write_file'}


### `list_schemas()` — what the model actually sees

The model never gets the python `Tool` dataclass. It gets `arcllm.types.Tool`,
which has just `name`, `description`, and `parameters`. That's the JSON-Schema
shape the provider's tool-use API expects.

In [9]:
schemas = registry.list_schemas()
print(f"schemas type: {type(schemas[0]).__module__}.{type(schemas[0]).__name__}")
print()
for s in schemas:
    print(f"name        : {s.name}")
    print(f"description : {s.description}")
    print(f"parameters  : {s.parameters}")

schemas type: arcllm.types.Tool

name        : search
description : Search the local index
parameters  : {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}


## 5. `SandboxConfig` — allowlist mode (deny-by-default when configured)

`SandboxConfig` is the executor's permission boundary. It has two knobs:

```python
@dataclass
class SandboxConfig:
    allowed_tools: list[str] | None = None
    check: Callable[[str, dict], Awaitable[tuple[bool, str]]] | None = None
```

- **`allowed_tools=None`** — no allowlist filter. Use this when you trust your
  registry contents and only want the `check` callback as a gate.
- **`allowed_tools=[...]`** — explicit allowlist. **Anything not in the list is
  denied.** Deny-by-default.
- **`check`** — async callback for per-arguments policy (e.g., "search is fine,
  but not for `password`"). Runs *after* the allowlist check.

Both knobs are optional and orthogonal. You can use either, both, or neither
(then the sandbox is a no-op pass-through).

Below: build a runnable executor wired to a `Sandbox` and exercise the allowlist.

In [10]:
from arcrun.events import EventBus
from arcrun.executor import execute_tool_call
from arcrun.registry import ToolRegistry
from arcrun.sandbox import Sandbox
from arcrun.state import RunState
from arcrun.types import SandboxConfig


def setup_run(tools, sandbox_config=None, tool_timeout=None):
    """Stand up the minimum infrastructure for execute_tool_call."""
    bus = EventBus(run_id="exec-demo")
    state = RunState(
        messages=[Message(role="user", content="go")],
        registry=ToolRegistry(tools=tools, event_bus=bus),
        event_bus=bus,
        run_id="exec-demo",
        tool_timeout=tool_timeout,
    )
    sandbox = Sandbox(config=sandbox_config, event_bus=bus)
    return state, sandbox, bus


print("Helper ready.")

Helper ready.


### 5a. Happy path — tool *is* in the allowlist

In [11]:
state, sandbox, bus = setup_run(
    tools=[search_tool],
    sandbox_config=SandboxConfig(allowed_tools=["search"]),
)

tc = ToolCall(id="tc-1", name="search", arguments={"query": "executor"})
result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"ok                 : {ok}")
print(f"tool_calls_made    : {state.tool_calls_made}")
print(f"result content     : {result_msg.content[0].content}")
print()
print("Events:")
for e in bus.events:
    if e.type.startswith("tool."):
        print(f"  {e.type:14s} {dict(e.data)}")

ok                 : True
tool_calls_made    : 1
result content     : Found 3 results for 'executor'

Events:
  tool.start     {'name': 'search', 'arguments': {'query': 'executor'}, 'args_digest': '3a730bcadaa072afdf7b19c32cad19d78336fd28de6323d2c3653a3bb37a4811', 'args_size': 20}
  tool.end       {'name': 'search', 'result_length': 30, 'duration_ms': 0.0021457672119140625, 'result_digest': 'ea2cb1eadc42b43def39b530ca912a4bce04d895f847708c8080ed1101d0b491', 'result_size': 30}


### 5b. Tool *not* in allowlist — denied, model sees the reason

The executor returns a `tool_result` message containing `Error: tool denied — ...`.
That's what the model sees on the next turn — it can read the denial and adjust
its plan instead of crashing the loop. `state.tool_calls_made` stays at 0 (denied
calls don't count).

In [12]:
state, sandbox, bus = setup_run(
    tools=[search_tool],
    sandbox_config=SandboxConfig(allowed_tools=["read_file"]),  # search NOT in list
)

tc = ToolCall(id="tc-2", name="search", arguments={"query": "anything"})
result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"ok              : {ok}")
print(f"tool_calls_made : {state.tool_calls_made}  (denied calls don't count)")
print(f"LLM-facing msg  : {result_msg.content[0].content}")
print()
print("tool.denied event:")
for e in bus.events:
    if e.type == "tool.denied":
        print(f"  {dict(e.data)}")

ok              : False
tool_calls_made : 0  (denied calls don't count)
LLM-facing msg  : Error: tool denied — search: not in allowed tools

tool.denied event:
  {'name': 'search', 'reason': 'search: not in allowed tools'}


## 6. `SandboxConfig` — custom checker for per-arguments policy

`allowed_tools` is coarse-grained — "this tool, yes/no". The `check` callback
gives you per-call control by inspecting the **arguments**. Common uses:

- "`read_file` is allowed, but not under `/etc/`"
- "`search` is fine, but not for `password` queries"
- "`http_fetch` only to allow-listed domains"

The callback is `async (tool_name, params) -> (allowed, reason)`. It runs **after**
the allowlist filter, so you can use both: allowlist for capability gating, checker
for argument inspection.

In [13]:
async def policy_check(tool_name, params):
    """Allow search, but block queries that look like credential lookups."""
    if tool_name == "search":
        q = params.get("query", "").lower()
        for forbidden in ("password", "api_key", "secret"):
            if forbidden in q:
                return False, f"queries containing {forbidden!r} are not allowed"
    return True, ""


config = SandboxConfig(
    allowed_tools=["search"],  # capability gate
    check=policy_check,  # argument-level gate
)

# Allowed query
state, sandbox, bus = setup_run(tools=[search_tool], sandbox_config=config)
tc = ToolCall(id="tc-ok", name="search", arguments={"query": "executor pipeline"})
_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"'executor pipeline' query : ok={ok}")

# Denied query
state, sandbox, bus = setup_run(tools=[search_tool], sandbox_config=config)
tc = ToolCall(id="tc-bad", name="search", arguments={"query": "admin password"})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"'admin password' query    : ok={ok}")
print(f"  reason -> {result_msg.content[0].content}")

'executor pipeline' query : ok=True
'admin password' query    : ok=False
  reason -> Error: tool denied — queries containing 'password' are not allowed


### Fail-safe: a buggy checker is treated as denial

If the `check` callback raises, the sandbox treats it as a denial — never as an
allow. That's the right default for a permission system: **failing closed beats
failing open**.

In [14]:
async def buggy_check(tool_name, params):
    raise RuntimeError("checker exploded")


state, sandbox, bus = setup_run(
    tools=[search_tool],
    sandbox_config=SandboxConfig(allowed_tools=["search"], check=buggy_check),
)
tc = ToolCall(id="tc-bug", name="search", arguments={"query": "anything"})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"ok       : {ok}")
print(f"reason   : {result_msg.content[0].content}")
print(f"event    : {next(e for e in bus.events if e.type == 'tool.denied').data['reason']}")

ok       : False
reason   : Error: tool denied — check callback error
event    : check callback error


## 7. `SandboxError` taxonomy — every subtype, when it fires

These exceptions live on the **container-execute** side of the sandbox surface
(see `arcrun.builtins.contained_execute`). They're re-exported at the package
top level so callers catch them precisely.

```
SandboxError                  ← base; catch this to handle "any sandbox failure"
├── SandboxUnavailableError   ← no Docker/Podman socket reachable
├── SandboxTimeoutError       ← container exceeded wall-clock budget
├── SandboxOOMError           ← container killed by OOM (exit code 137)
└── SandboxRuntimeError       ← tool-level rule violated (oversized payload, etc.)
```

The container code itself is covered in `arcrun/03-codeexec.ipynb`. Here we
focus on the **error class hierarchy** — when each fires, and how to catch them.
Each demo below intentionally raises and catches the exception so you can see
the full taxonomy without needing Docker installed.

In [15]:
from arcrun import (
    SandboxError,
    SandboxOOMError,
    SandboxRuntimeError,
    SandboxTimeoutError,
    SandboxUnavailableError,
)

# Class hierarchy: every leaf subclasses SandboxError
for cls in (
    SandboxUnavailableError,
    SandboxTimeoutError,
    SandboxOOMError,
    SandboxRuntimeError,
):
    print(f"{cls.__name__:25s} <- {cls.__mro__[1].__name__}")

SandboxUnavailableError   <- SandboxError
SandboxTimeoutError       <- SandboxError
SandboxOOMError           <- SandboxError
SandboxRuntimeError       <- SandboxError


### 7a. `SandboxUnavailableError` — no container runtime found

Fires inside `_detect_socket()` when no Docker/Podman socket is reachable. Common
in CI without docker, or if the daemon isn't running.

```python
# this raises SandboxUnavailableError
```

In [16]:
try:
    raise SandboxUnavailableError("No container runtime socket found. Install Docker or Podman.")
except SandboxError as e:
    print(f"denied: {type(e).__name__}: {e}")
    print(f"isinstance(e, SandboxError) -> {isinstance(e, SandboxError)}")

denied: SandboxUnavailableError: No container runtime socket found. Install Docker or Podman.
isinstance(e, SandboxError) -> True


### 7b. `SandboxTimeoutError` — container exceeded wall-clock budget

Fires when the inner `asyncio.wait_for` around the container exec times out.

```python
# this raises SandboxTimeoutError
```

In [17]:
try:
    raise SandboxTimeoutError("Execution exceeded 30s timeout")
except SandboxTimeoutError as e:
    print(f"denied: {type(e).__name__}: {e}")
    # SandboxError is the base — broad catches still work
    print(f"caught as SandboxError? {isinstance(e, SandboxError)}")

denied: SandboxTimeoutError: Execution exceeded 30s timeout
caught as SandboxError? True


### 7c. `SandboxOOMError` — container killed (exit 137)

Linux's OOM killer sends SIGKILL when a container exceeds its memory limit. The
container exits with code 137 (= 128 + 9). The runtime checks for that exact
exit and converts it to a structured error.

```python
# this raises SandboxOOMError
```

In [18]:
try:
    raise SandboxOOMError("Container killed by OOM (exit code 137)")
except SandboxOOMError as e:
    print(f"denied: {type(e).__name__}: {e}")

denied: SandboxOOMError: Container killed by OOM (exit code 137)


### 7d. `SandboxRuntimeError` — tool-level rule violated

Fires for "the call is structurally bad" cases that aren't timeouts or OOM. The
canonical one in `contained_execute.py` is the **code-size guard** — the model
shipped a payload larger than `MAX_CODE_BYTES` (1 MiB).

```python
# this raises SandboxRuntimeError
```

In [19]:
MAX_CODE_BYTES = 1_048_576
oversized = 2 * MAX_CODE_BYTES

try:
    raise SandboxRuntimeError(f"Code size {oversized} exceeds {MAX_CODE_BYTES} byte limit")
except SandboxRuntimeError as e:
    print(f"denied: {type(e).__name__}: {e}")

denied: SandboxRuntimeError: Code size 2097152 exceeds 1048576 byte limit


### 7e. Catching the whole family

Most callers want one block: "if anything sandbox-y goes wrong, handle it." That's
why `SandboxError` exists — catch the base.

In [20]:
def simulate(error):
    try:
        raise error
    except SandboxError as e:
        return f"caught {type(e).__name__}: {e}"


for err in (
    SandboxUnavailableError("no socket"),
    SandboxTimeoutError("30s exceeded"),
    SandboxOOMError("OOM at 256MiB"),
    SandboxRuntimeError("payload too large"),
):
    print(simulate(err))

caught SandboxUnavailableError: no socket
caught SandboxTimeoutError: 30s exceeded
caught SandboxOOMError: OOM at 256MiB
caught SandboxRuntimeError: payload too large


## 8. Classification — `read_only` vs `state_modifying`

Why classify? Because **read-only tool calls are safe to dispatch in parallel**
and **anything else must run sequentially** (or risk write-then-read races).
That's the engine of the parallel dispatch optimization.

There are two layers:

1. **`Tool.classification: str`** — the per-tool flag a developer sets when
   defining the tool. Set to `"read_only"` for pure reads (`grep`, `ls`, `read_file`),
   `"state_modifying"` for anything that mutates (`write_file`, `bash -c rm`, `http POST`).
2. **`ClassificationRegistry` Protocol** (in `arcrun.parallel_dispatch`) — what
   the dispatcher actually consults. It expects a `get_classification(name) -> str`
   method returning `"read_only"` or `"state_modifying"`. Any object that
   implements that one method works — arcrun deliberately doesn't depend on
   `arcagent`'s registry.

The `BatchClassifier` uses the registry to partition incoming batches. Below we
build a tiny `ClassificationRegistry` adapter and run the classifier — without
running any tools — so you can see the verdict logic.

In [21]:
from arcrun.parallel_dispatch import BatchClassifier, BatchVerdict


class _ClassRegistry:
    """Minimal ClassificationRegistry: maps name -> classification string."""

    def __init__(self, classes: dict[str, str]) -> None:
        self._classes = classes

    def get_classification(self, name: str) -> str:
        # Unknown tools are conservatively treated as state_modifying (fail-closed).
        return self._classes.get(name, "state_modifying")


classes = _ClassRegistry(
    {
        "search": "read_only",
        "read_file": "read_only",
        "write_file": "state_modifying",
    }
)
print(f"search      -> {classes.get_classification('search')}")
print(f"read_file   -> {classes.get_classification('read_file')}")
print(f"write_file  -> {classes.get_classification('write_file')}")
print(f"unknown_x   -> {classes.get_classification('unknown_x')}  (fail-closed default)")

search      -> read_only
read_file   -> read_only
write_file  -> state_modifying
unknown_x   -> state_modifying  (fail-closed default)


### Verdict: all read-only -> parallelizable

In [22]:
classifier = BatchClassifier(classes)

batch_a = [
    ToolCall(id="a", name="search", arguments={"query": "x"}),
    ToolCall(id="b", name="read_file", arguments={"path": "/data/a.txt"}),
    ToolCall(id="c", name="read_file", arguments={"path": "/data/b.txt"}),
]
verdict = classifier.classify(batch_a)
print(f"can_parallelize : {verdict.can_parallelize}")
print(f"reason          : {verdict.reason}")

can_parallelize : True
reason          : all_read_only


### Verdict: any state-modifying tool -> sequential

In [23]:
batch_b = [
    ToolCall(id="a", name="search", arguments={"query": "x"}),
    ToolCall(id="b", name="write_file", arguments={"path": "/data/a.txt"}),
]
verdict = classifier.classify(batch_b)
print(f"can_parallelize : {verdict.can_parallelize}")
print(f"reason          : {verdict.reason}")

can_parallelize : False
reason          : state_modifying tool in batch: write_file


### Verdict: implicit dependency (shared path arg) -> sequential

Even when every tool is read-only, the classifier scans for argument values that
look like paths (contain `/` or `\`). If two calls share a value, the batch
runs sequential — to avoid a write that snuck in between identical-looking
read-only calls in the audit log. False positives only cost parallelism; false
negatives would cause a race.

In [24]:
batch_c = [
    ToolCall(id="a", name="read_file", arguments={"path": "/data/shared.txt"}),
    ToolCall(id="b", name="search", arguments={"query": "/data/shared.txt"}),
]
verdict = classifier.classify(batch_c)
print(f"can_parallelize : {verdict.can_parallelize}")
print(f"reason          : {verdict.reason}")

can_parallelize : False
reason          : implicit dependency: shared path argument across calls


### Why this matters for the executor

The executor itself is **sequential** — `execute_tool_call()` does one call at
a time. Parallelism is a layer above: a batch arrives, the classifier decides
parallel-vs-sequential, and a `ParallelDispatcher` or `SequentialDispatcher`
fans out the calls — each of which still goes through the same single-call
executor.

That separation is intentional: the executor stays small and total, the
parallelism layer can evolve independently, and audit ordering is preserved
regardless of completion order via the `_seq` arguments-key the dispatcher
injects.

For the deep dive on `BatchClassifier`, `ParallelDispatcher`, semaphore tuning,
FIPS mode caps, race-free audit ordering, and benchmarks — see
**`arcrun/05-parallel-dispatch.ipynb`**.

## 9. Errors and retries — what happens when a tool raises

The executor's contract for failure is:

- The exception is **caught** — the loop never crashes from a tool error.
- A `tool.error` event is emitted with the **full** exception detail.
- The model gets a **truncated** copy (200 chars) so wide stack traces don't
  poison the context window.
- `state.tool_calls_made` is **not** incremented (only successes count).
- The strategy gets `(error_message, False)` and decides whether to retry.

That last point is important: **the executor does not retry**. Retries are a
strategy concern. ReAct's posture is "show the model the error, let it choose
whether to try a different approach." Other strategies might wrap calls in
exponential backoff; either way, the executor is just the inner step.

### 9a. Tool raises an exception — caught, surfaced as `tool.error`

In [25]:
async def _explode(params, ctx):
    raise ConnectionError("database refused connection")


bomb = Tool(
    name="bomb",
    description="Always raises",
    input_schema={"type": "object"},
    execute=_explode,
)

state, sandbox, bus = setup_run(tools=[bomb])
tc = ToolCall(id="tc-bomb", name="bomb", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)

print(f"ok                 : {ok}")
print(f"tool_calls_made    : {state.tool_calls_made}  (failures don't count)")
print(f"LLM-facing message : {result_msg.content[0].content}")
print()
print("Events:")
for e in bus.events:
    if e.type.startswith("tool."):
        print(f"  {e.type:14s} {dict(e.data)}")

ok                 : False
tool_calls_made    : 0  (failures don't count)
LLM-facing message : Error: ConnectionError: database refused connection

Events:
  tool.start     {'name': 'bomb', 'arguments': {}, 'args_digest': '44136fa355b3678a1146ad16f7e8649e94fb4fc21fe77e8310c060f61caaff8a', 'args_size': 2}
  tool.error     {'name': 'bomb', 'error': 'database refused connection', 'args_digest': '44136fa355b3678a1146ad16f7e8649e94fb4fc21fe77e8310c060f61caaff8a', 'args_size': 2}


### 9b. Schema validation runs *before* `execute()` — your code never sees bad args

The executor calls `jsonschema.validate(args, tool.input_schema)` at step 4.
If the model invents a field, omits a required one, or sends the wrong type,
the executor returns an `Error: invalid params — ...` message **before** your
tool runs.

In [26]:
state, sandbox, bus = setup_run(tools=[search_tool])

# Wrong type: query is supposed to be a string, model sent an int
tc = ToolCall(id="tc-bad", name="search", arguments={"query": 42})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"wrong type       : ok={ok}")
print(f"  -> {result_msg.content[0].content}")

# Missing required field
state, sandbox, bus = setup_run(tools=[search_tool])
tc = ToolCall(id="tc-missing", name="search", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"missing required : ok={ok}")
print(f"  -> {result_msg.content[0].content}")

wrong type       : ok=False
  -> Error: invalid params — 42 is not of type 'string'
missing required : ok=False
  -> Error: invalid params — 'query' is a required property


### 9c. Tool not found — registry miss

If the model invents a tool name that's not in the registry, the executor
returns `Error: tool '<name>' not found`. The model can read that and pick
something it actually has.

In [27]:
state, sandbox, bus = setup_run(tools=[search_tool])
tc = ToolCall(id="tc-ghost", name="non_existent_tool", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"ok      : {ok}")
print(f"message : {result_msg.content[0].content}")

ok      : False
message : Error: tool 'non_existent_tool' not found


### 9d. Per-tool timeout — `Tool.timeout_seconds` wins over `state.tool_timeout`

```
timeout = tool_def.timeout_seconds or state.tool_timeout   # executor.py:58
```

Priority: per-tool > state-default > none.

In [28]:
async def _slow(params, ctx):
    await asyncio.sleep(10)
    return "done"


# Per-tool timeout — wins
slow_tool = Tool(
    name="slow",
    description="Sleeps 10s",
    input_schema={"type": "object"},
    execute=_slow,
    timeout_seconds=0.05,
)
state, sandbox, bus = setup_run(tools=[slow_tool])
tc = ToolCall(id="tc-slow1", name="slow", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"per-tool timeout  : ok={ok}, msg={result_msg.content[0].content}")

# Global timeout — used when per-tool not set
no_timeout_tool = Tool(
    name="slow",
    description="Sleeps 10s, no per-tool timeout",
    input_schema={"type": "object"},
    execute=_slow,
)
state, sandbox, bus = setup_run(tools=[no_timeout_tool], tool_timeout=0.05)
tc = ToolCall(id="tc-slow2", name="slow", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)
print(f"global timeout    : ok={ok}, msg={result_msg.content[0].content}")

per-tool timeout  : ok=False, msg=Error: tool timed out after 0.05s
global timeout    : ok=False, msg=Error: tool timed out after 0.05s


### 9e. Error-message truncation — full detail in audit, short copy to the LLM

Long tracebacks waste context tokens, but throwing them away breaks debugging.
The executor splits the difference: the LLM sees ≤ 200 chars (`_MAX_ERROR_LEN`),
the `tool.error` event keeps the full message.

In [29]:
async def _verbose_error(params, ctx):
    raise RuntimeError("x" * 500)  # 500-char error


verbose = Tool(
    name="verbose",
    description="Raises a 500-char error",
    input_schema={"type": "object"},
    execute=_verbose_error,
)
state, sandbox, bus = setup_run(tools=[verbose])
tc = ToolCall(id="tc-v", name="verbose", arguments={})
result_msg, ok = await execute_tool_call(tc, state, sandbox)

llm_text = result_msg.content[0].content
audit_text = next(e for e in bus.events if e.type == "tool.error").data["error"]

print(f"LLM sees   : {len(llm_text)} chars")
print(f"Audit log  : {len(audit_text)} chars")
print()
print(f"LLM message head : {llm_text[:80]}...")
print(f"Audit message ends with: ...{audit_text[-12:]}")

LLM sees   : 221 chars
Audit log  : 500 chars

LLM message head : Error: RuntimeError: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx...
Audit message ends with: ...xxxxxxxxxxxx


### 9f. End-to-end through `run()` — every event in order

To see the full pipeline composing, drive a `run()` with `MockModel` that
emits one `tool_use` and then ends the turn. The events list reads like a
storyboard.

In [30]:
from arcrun import StaticProvider
from arcrun.loop import run

model = MockModel(
    [
        # Turn 1: model calls search
        LLMResponse(
            content="Searching...",
            tool_calls=[ToolCall(id="tc1", name="search", arguments={"query": "executor"})],
            stop_reason="tool_use",
        ),
        # Turn 2: model finishes
        LLMResponse(content="All set.", stop_reason="end_turn"),
    ]
)

events_seen = []

result = await run(
    model=model,
    capabilities=StaticProvider([search_tool]),
    system_prompt="You are a helpful assistant.",
    task="Search for executor docs",
    on_event=events_seen.append,
)

print(f"content   : {result.content}")
print(f"turns     : {result.turns}")
print(f"tool calls: {result.tool_calls_made}")
print()
print("Event trail (tool.* events only):")
for e in result.events:
    if e.type.startswith("tool."):
        print(f"  {e.type:18s} {dict(e.data)}")

content   : All set.
turns     : 2
tool calls: 1

Event trail (tool.* events only):
  tool.start         {'name': 'search', 'arguments': {'query': 'executor'}, 'args_digest': '3a730bcadaa072afdf7b19c32cad19d78336fd28de6323d2c3653a3bb37a4811', 'args_size': 20}
  tool.end           {'name': 'search', 'result_length': 30, 'duration_ms': 0.00095367431640625, 'result_digest': 'ea2cb1eadc42b43def39b530ca912a4bce04d895f847708c8080ed1101d0b491', 'result_size': 30}


## 10. Pointers — what's covered elsewhere

Two adjacent topics deliberately live in their own notebooks so this one stays
focused on the executor surface:

### `arcrun/05-parallel-dispatch.ipynb` — concurrent dispatch deep dive
- `BatchClassifier` rules end-to-end, including the implicit-dependency heuristic
- `ParallelDispatcher` and `SequentialDispatcher`
- Semaphore tuning (`max_parallel`), FIPS mode (cap=4)
- Submission-order preservation via the `_seq` arguments-key
- Race-free audit ordering and benchmarks

### `arcrun/03-codeexec.ipynb` — code execution strategy & container isolation
- `make_execute_tool` (in-process sandbox)
- `make_contained_execute_tool` (Docker isolation, mem/cpu/network limits)
- The CodeExec strategy and `prompt_guidance`
- Where each `SandboxError` subtype actually fires inside the container path

If you wanted to know *why* the classifier matters, you saw it in §8 above.
The deep dive is in `arcrun/05-parallel-dispatch.ipynb`. If you wanted to
know *how* the container path enforces its limits, the wiring is in
`arcrun/03-codeexec.ipynb`.

## 11. Summary

The tool executor is the smallest, most-used piece of arcrun. Every strategy —
ReAct, CodeExec, anything you write — runs every single tool call through it.
That's why it has exactly one job and ~85 lines of code.

**Pipeline (10 steps):**
1. emit `tool.start`
2. `Sandbox.check()` — deny? return `tool.denied` event + error message
3. `ToolRegistry.get(name)` — miss? return error message
4. `jsonschema.validate(args, schema)` — fail? return error message
5. build `ToolContext`
6. `await tool.execute(args, ctx)` (with optional timeout)
7. catch exceptions → emit `tool.error`, return truncated error
8. emit `tool.end` with `duration_ms`
9. `state.tool_calls_made += 1`
10. return `(tool_result_message, True)`

**Public surface this notebook covered:**
- `arcrun.ToolRegistry` — `add`, `remove`, `get`, `list_schemas`, `names`
- `arcrun.Tool` — `name`, `description`, `input_schema`, `execute`, `timeout_seconds`, `classification`, `signals_completion`
- `arcrun.ToolContext` — `run_id`, `tool_call_id`, `turn_number`, `event_bus`, `cancelled`, `parent_state`
- `arcrun.SandboxConfig` — allowlist mode and the `check` callback
- `arcrun.SandboxError`, `arcrun.SandboxUnavailableError`, `arcrun.SandboxTimeoutError`, `arcrun.SandboxOOMError`, `arcrun.SandboxRuntimeError`
- `arcrun.parallel_dispatch.BatchClassifier`, `BatchVerdict`, `ClassificationRegistry` Protocol
- `arcrun.executor.execute_tool_call` (the pipeline itself)
- `arcrun.run` (used end-to-end in §9f to show the full event trail)

**Where to go next:**
- **Deep dive on parallel dispatch** → `arcrun/05-parallel-dispatch.ipynb`
- **Container execution and the CodeExec strategy** → `arcrun/03-codeexec.ipynb`
- **The full ReAct loop with steering, follow-up, and event-chain verification** → `arcrun/01-core-react.ipynb`

**Key insight:** The executor doesn't retry, doesn't compose, doesn't decide
parallelism. It walks one call through one pipeline. Everything else — retry
policies, parallel batching, container isolation, strategy choice — is layered
on top, which is why each of those concerns can evolve without touching the
executor itself.